# Module 11 · 6：生成式與多模態大模型訓練藍圖

> **本節定位｜總結藍圖（概念為主）**
> 前面幾節做的是「**判別式**」下游（分類/辨識/檢索）。本節拉高視角，
> 概覽 2026 兩大前沿：**生成式模型（LLM、Diffusion）** 與 **多模態大模型 (VLM)**
> 的資料格式與訓練管線，把整個課程收束成一張大圖。

## 學習目標
- 認識 **LLM 三階段訓練管線**：預訓練 → SFT → 偏好對齊。
- 認識 **文生圖 Diffusion** 的資料格式與訓練概念。
- 認識 **VLM（視覺語言模型）** 如何把視覺接進 LLM。

## 1. LLM 訓練管線：Pretrain → SFT → 偏好對齊

```
海量純文字          指令/對話資料            人類偏好資料
{"text": ...}      {"messages": [...]}     {prompt, chosen, rejected}
     │                   │                        │
     ▼                   ▼                        ▼
┌──────────┐      ┌──────────┐            ┌──────────────┐
│ 預訓練    │ ───► │ SFT 微調  │ ─────────► │ 偏好對齊       │
│ Pretrain │      │ (指令遵循) │            │ DPO / RLHF    │
└──────────┘      └──────────┘            └──────────────┘
 學語言/知識        學會聽指令               對齊人類偏好/安全
```

| 階段 | 資料格式（對應 M09 文字 04） | 目標 | 常用工具 |
|:--|:--|:--|:--|
| 預訓練 | `{"text": ...}` + packing | 學語言與世界知識 | transformers |
| SFT | chat/instruction JSONL | 學會遵循指令 | `trl` SFTTrainer + `peft` |
| 偏好對齊 | `{prompt, chosen, rejected}` | 對齊人類偏好 | `trl` DPOTrainer |

**資料工程貫穿全程**：去重、品質過濾、去污染（M09 文字 04）決定了模型上限。

<!-- concept-image:06_generative_and_multimodal_blueprint_fig1 -->
<div align="center">
<img src="concept_images/06_generative_multimodal_fig1_llm_pipeline_20260611_225248.png" alt="LLM 三階段訓練管線" width="640" style="max-width:100%;">
<br><sub>圖 1 · LLM 三階段訓練管線</sub>
</div>

## 2. 文生圖 Diffusion

| 項目 | 內容 |
|:--|:--|
| 資料格式 | **圖文配對** `{image, caption}`（與 CLIP 類似，見 M09 多模態 01） |
| 前處理 | 影像 → VAE 編成 latent；caption → text encoder |
| 訓練目標 | 學習從「加噪的 latent」**還原原圖**（去噪），並以文字為條件 |
| 微調技巧 | LoRA、DreamBooth（少量圖客製風格/主體） |

直覺：把資料前處理學到的「圖像張量化 + 文字 tokenize」組合起來，
就是訓練文生圖模型的資料基礎。

<!-- concept-image:06_generative_and_multimodal_blueprint_fig2 -->
<div align="center">
<img src="concept_images/06_generative_multimodal_fig2_diffusion_20260611_225353.png" alt="文生圖 Diffusion 訓練架構" width="640" style="max-width:100%;">
<br><sub>圖 2 · 文生圖 Diffusion 訓練架構</sub>
</div>

## 3. 多模態大模型 (VLM)

```
  影像 ──► 視覺編碼器(ViT/CLIP) ──► 投影層 ──┐
                                             ├──► LLM ──► 文字輸出
  文字 ──────────► tokenizer ──────────────►┘
```

| 項目 | 內容 |
|:--|:--|
| 資料格式 | 帶圖對話 messages（image+text，見 M09 多模態 01） |
| 關鍵設計 | 影像 → 「視覺 token」，與文字 token **串接**後一起進 LLM |
| 訓練 | 通常凍結視覺編碼器，先訓投影層，再 SFT 整體 |
| 代表 | LLaVA、Qwen-VL、PaliGemma |

<!-- concept-image:06_generative_and_multimodal_blueprint_fig3 -->
<div align="center">
<img src="concept_images/06_generative_multimodal_fig3_vlm_fusion_20260611_225447.png" alt="多模態大模型 (VLM) 架構與融合" width="640" style="max-width:100%;">
<br><sub>圖 3 · 多模態大模型 (VLM) 架構與融合</sub>
</div>

## 4. 全課程收束：資料前處理 → 大模型

| 模態 | M09 學到的前處理 | M11 能訓練的模型 |
|:--|:--|:--|
| 文字 | tokenizer、嵌入、LLM 資料格式 | 分類器、LLM(LoRA/SFT)、RAG |
| 圖像 | 張量化、ViT/CLIP、增強 | ViT 分類、CLIP、Diffusion |
| 聲音 | 16k/log-mel、特徵抽取 | Whisper ASR、wav2vec2 分類 |
| 影片 | 抽樣、`(N,T,C,H,W)` | VideoMAE 動作辨識 |
| 多模態 | 圖文配對、VLM 格式 | CLIP、VLM |

**核心結論**：**「資料結構設計」是一切的起點**——形狀與格式決定能接哪種模型；
把資料整理對了，2026 的大模型訓練/微調，大多是「換模型、套同一套 Trainer/PEFT 流程」。

## 小結
- LLM：預訓練 → SFT → 偏好對齊；資料工程決定上限。
- Diffusion：圖文配對 + 去噪訓練。
- VLM：視覺 token 接進 LLM。
- 全課程主軸：**資料結構設計 → 對應模型 → 訓練**。